<a href="https://colab.research.google.com/github/Bryan-625/DS340W-Project/blob/main/FinGPT_Inference_Llama2_13B_falcon_7B_for_Beginners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers==4.32.0 peft==0.5.0
!pip install sentencepiece
!pip install accelerate
!pip install torch
!pip install peft
!pip install datasets
!pip install bitsandbytes
!pip install -q -U torchao

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 25.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.4 MB/s  0:00:00
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokenizers
error: failed-wheel-build-for-install

× Failed to build installable wheels for some pyproject.toml based projects
╰─> tokenizers
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.1 MB/s  0:00:03


In [ ]:
#for the `load_in_8bit=True` error
# !pip install protobuf transformers==4.32.0 cpm_kernels torch>=2.0 gradio mdtex2html sentencepiece accelerate

## Inference with Single Task: Sentiment

In [3]:
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, LlamaForCausalLM, LlamaTokenizerFast, BitsAndBytesConfig
from peft import PeftModel  # 0.5.0

W1114 22:15:39.567000 6496 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1114 22:15:39.609000 6496 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [4]:
# Load Models
base_model = "NousResearch/Llama-2-13b-hf"
peft_model = "FinGPT/fingpt-sentiment_llama2-13b_lora"
tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
quant_config = BitsAndBytesConfig(load_in_8bit=True)
model = LlamaForCausalLM.from_pretrained(base_model, trust_remote_code=True, device_map = "cuda:0", quantization_config=quant_config,)
model = PeftModel.from_pretrained(model, peft_model)
model = model.eval()

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/33.4k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/196 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

adapter_model.bin: reconstructing file:   0%|          |  0.00B / 19.7MB            

adapter_model.bin: downloading bytes:           |  0.00B            

In [5]:
# Make prompts
prompt = [
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is aggressively pursuing its growth strategy by increasingly focusing on technologically more demanding HDI printed circuit boards PCBs .
Answer: ''',
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .
Answer: ''',
'''Instruction: What is the sentiment of this news? Please choose an answer from {negative/neutral/positive}
Input: A tinyurl link takes users to a scamming site promising that users can earn thousands of dollars by becoming a Google ( NASDAQ : GOOG ) Cash advertiser .
Answer: ''',
]

# Generate results
tokens = tokenizer(prompt, return_tensors='pt', padding=True, max_length=512)
res = model.generate(**tokens, max_length=512)
res_sentences = [tokenizer.decode(i) for i in res]
out_text = [o.split("Answer: ")[1] for o in res_sentences]

# show results
for sentiment in out_text:
    print(sentiment)

# Output:
# positive
# neutral
# negative


/usr/local/lib/python3.13/dist-packages/transformers/tokenization_utils_base.py:2387: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


 positive</s>
 neutral</s>
 neutral</s>


## New Implementation

In [2]:
# Loading Libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, LlamaTokenizerFast, LlamaForCausalLM, BitsAndBytesConfig, AutoModelForSequenceClassification
import torch
from peft import PeftModel

W1114 22:23:52.565000 675 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1114 22:23:52.607000 675 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [3]:
# Load FinGPT Model (Llama-2-13b + LoRA Adapter)

base_model = "NousResearch/Llama-2-13b-hf"
peft_model = "FinGPT/fingpt-sentiment_llama2-13b_lora"

tokenizer = LlamaTokenizerFast.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(load_in_8bit=True)

fingpt_model = LlamaForCausalLM.from_pretrained(
    base_model,
    trust_remote_code=True,
    device_map="cuda:0",
    quantization_config=quant_config,
)
fingpt_model = PeftModel.from_pretrained(fingpt_model, peft_model)
fingpt_model = fingpt_model.eval()

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/631 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/33.4k [00:00<?, ?B/s]

... (3 outputs left out)

generation_config.json:   0%|          | 0.00/196 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

adapter_model.bin: reconstructing file:   0%|          |  0.00B / 19.7MB            

adapter_model.bin: downloading bytes:           |  0.00B            

In [5]:
# Load Web Filter Model (Used to filter noise in retrieved documents)

# WebFilter classifies retrieved web documents as 'Reliable/Relevant' vs 'Unreliable/Irrelevant'
web_filter_model_name = "dayll/WebFilter-7B"
filter_tokenizer = AutoTokenizer.from_pretrained(web_filter_model_name, trust_remote_code=True)
filter_model = AutoModelForSequenceClassification.from_pretrained(
    web_filter_model_name,
    device_map="cuda:0"
)
filter_model = filter_model.eval()

config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: dayll/WebFilter-7B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
# Web Filter RAG Pipeline

def filter_retrieved_documents(query, retrieved_docs, threshold=0.5):
    """
    Filters out noisy or irrelevant web documents using the WebFilter classifier.
    """
    filtered_docs = []
    for doc in retrieved_docs:
        # Format input similar to standard WebFilter prompt templates: [Query] [Doc]
        inputs = filter_tokenizer(
            f"Query: {query} Document: {doc}",
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to("cuda")

        with torch.no_grad():
            outputs = filter_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)
            # WebFilter typically has 2 classes: 0 (noise/irrelevant), 1 (clean/relevant)
            relevance_score = probs[0][1].item()

        if relevance_score >= threshold:
            filtered_docs.append(doc)

    return filtered_docs

def fingpt_rag_query(query, raw_retrieved_docs):
    # Filter the noisy web search documents
    clean_docs = filter_retrieved_documents(query, raw_retrieved_docs)

    # Combine clean documents into a consolidated context
    context = "\n".join([f"- {doc}" for doc in clean_docs]) if clean_docs else "No relevant web context found."

    # Construct FinGPT prompt incorporating clean context
    prompt = f"""Instruction: Analyze the financial sentiment or answer the query based on the clean web context.
Context:
{context}

Query: {query}
Answer: """

    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = fingpt_model.generate(**inputs, max_length=512)

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

# Example usage:
# raw_web_results = [
#     "Apple Inc. announces record-breaking Q3 earnings with a 15% increase in hardware sales.",
#     "CLICK HERE TO WIN A FREE IPHONE 15 NOW!!! Cheap stock tips on blogspot!"
# ]
# print(fingpt_rag_query("What is the sentiment regarding Apple's latest hardware sales?", raw_web_results))